# Reuters

The News data set that I found was insufficient as the date ranges were from 2017- July 2020.

Collecting from the beginning of 2020 until the current day will allow me to make year-on-year comparisons.

Legal information:

_"It is provided by Reuters and its licensors to you for your personal use and information only. You may not use the Content or Service for any commercial purpose. You may not remove, alter, forward, scrape, copy, sell, distribute, retransmit, create derivative works or otherwise make available the Content to third parties without our prior written consent,"_

From my understanding it seems that it is fine for personal useage to use this information.

Building my model and using it personally = valid

Offering this model to third parites = against terms of service

In [57]:
import requests
from bs4 import BeautifulSoup
from datetime import date

# Plan is to itterate as far back as needed
url_prefix = 'https://www.reuters.com/news/archive/worldnews?view=page&page='
page_num = 1
url_suffix = '&pageSize=10'

In [7]:
def get_url(page_number):
    return url_prefix + str(page_num)  + url_suffix

get_url(1) # Link works

'https://www.reuters.com/news/archive/worldnews?view=page&page=1&pageSize=10'

In [9]:
r = requests.get(get_url(1))
soup = BeautifulSoup(r.content, 'html.parser')
# print(soup.prettify())

## Getting the data required

The list of headlines are located: "class="news-headline-list"

And are wrapped in:
			<article class="story ">

The data that I want is:

							<h3 class="story-title">
								Iran says Natanz nuclear site hit by terrorism - TV</h3>
							</a>
			        <div class="contributor"></div>
			        <p>DUBAI (Reuters) -An incident at Iran's Natanz nuclear facility on Sunday was caused by an act of "nuclear terrorism", the country's nuclear chief Ali Akbar Salehi said, according to state TV, adding that Tehran reserves the right to take action against the perpetrators.</p>
					<time class="article-time">
							<span class="timestamp">12:57pm EDT</span>


The timestamp contains just the time of publishing if on the current day, otherwise it is in the form of:

			<time class="article-time">
				<span class="timestamp">Apr 10 2021</span>
			</time>

In [18]:
a_story = stories[0]
title = a_story.get('h3')
description = a_story.get('p')
published = a_story.find_all('span', class_='timestamp')[0]
print("title: " + title)
print("description: " + description)
print("published: " + published)

<article class="story">
<div class="story-photo lazy-photo">
<a href="/article/us-health-coronavirus-netherlands/dutch-lockdown-measures-remain-until-at-least-april-28-anp-says-idUSKBN2BY0LN">
<img alt="" border="0" org-src="https://s3.reutersmedia.net/resources/r/?m=02&amp;d=20210411&amp;t=2&amp;i=1558046613&amp;w=200&amp;fh=&amp;fw=&amp;ll=&amp;pl=&amp;sq=&amp;r=LYNXMPEH3A0CP" src="https://s1.reutersmedia.net/resources_v2/images/1x1.png"/>
</a>
</div><div class="story-content">
<a href="/article/us-health-coronavirus-netherlands/dutch-lockdown-measures-remain-until-at-least-april-28-anp-says-idUSKBN2BY0LN">
<h3 class="story-title">
								Dutch lockdown measures remain until at least April 28, ANP says</h3>
</a>
<div class="contributor"></div>
<p>    The Dutch government on Sunday dashed hopes of an early easing of lockdown, saying a night-time curfew and other restrictions would remain until at least April 28 as daily infections rose to a two-week high.</p>
<time class="article-tim

In [49]:
def find_title(story):
    try:
        return story.find_all('h3', class_='story-title')[0].getText().strip()
    except:
        return 'no_title'
    
find_title(a_story)

'Dutch lockdown measures remain until at least April 28, ANP says'

In [71]:
def find_description(story):
    try:
        return story.find_all('p')[0].getText().strip()
    except:
        return 'no_description'
    
print("description: " + find_description(a_story))

description: The Dutch government on Sunday dashed hopes of an early easing of lockdown, saying a night-time curfew and other restrictions would remain until at least April 28 as daily infections rose to a two-week high.


In [70]:
def find_published_date(story):
    try:
        pub_date = story.find_all('span', class_='timestamp')[0].getText().strip()
        # Date is in the form APR 10 2021
        if 'EDT' in pub_date:
            pub_date = date.today().strftime('%b %d %Y')
        return pub_date
    except:
        return 'no_date'

print("published: " + find_published_date(a_story))

published: Apr 11 2021


In [88]:
def extract_article_information_dict(story):
    return {
        "title": find_title(story),
        "description": find_description(story),
        "date": find_published_date(story)
    }

In [82]:
def extract_article_information_list(story):
    return [find_title(story), find_description(story), find_published_date(story)]

In [91]:
print(extract_article_information_dict(a_story))

{'title': 'Dutch lockdown measures remain until at least April 28, ANP says', 'description': 'The Dutch government on Sunday dashed hopes of an early easing of lockdown, saying a night-time curfew and other restrictions would remain until at least April 28 as daily infections rose to a two-week high.', 'date': 'Apr 11 2021'}


In [96]:

mock_list = []
mock_list.append(extract_article_information_dict(stories[0]))
mock_list.append(extract_article_information_dict(stories[1]))
mock_list.append(extract_article_information_dict(stories[2]))
mock_list

#(columns=['title', 'description', 'date'])
mock_results = pd.DataFrame.from_dict(mock_list)
mock_results

,title,description,date
0,Dutch lockdown measures remain until at least ...,The Dutch government on Sunday dashed hopes of...,Apr 11 2021
1,Iran says Natanz nuclear site hit by terrorism...,DUBAI (Reuters) -An incident at Iran's Natanz ...,Apr 11 2021
2,Ecuador chooses its economic future in preside...,QUITO (Reuters) -Ecuadoreans voted in a presid...,Apr 11 2021


In [99]:
def extract_articles(reuters_url):
    """
    Returns a list of articles from the given link, broken down into an object with key {'title', 'description', 'date'])}
    If there the page can not be properly loaded then an empty list is returned
    """
    results = []
    
    try:
        r = requests.get(reuters_url)
        soup = BeautifulSoup(r.content, 'html.parser')
    except:
        print('The page could not be parsed: ' + reuters_url)
        return results
        
    for article in soup.find_all('article'):
        try: 
            details = extract_article_information_dict(article)
            results.append(details)
        except:
            print('Information for an article at URL: ' + reuters_url + 'could not be extracted')
    
    return results

In [98]:
results_list = []

In [113]:
# Hopefully I won't get blocked.... massively slowed after page 40
# Looooooong pause at 803
# Could have also checked if the date representation of the string is 2019


#for page_num in range(1,1000):
#    if(page_num % 100): # forgot to set an equality
#        print('extracting data from page: ' + str(page_num))
#    url = get_url(page_num)
#    results_list.extend(extract_articles(url))

In [100]:
a = []
a.append(extract_article_information_dict(stories[0]))
a.append(extract_article_information_dict(stories[1]))
a.append(extract_article_information_dict(stories[2]))

b = []
b.append(extract_article_information_dict(stories[3]))
b.append(extract_article_information_dict(stories[4]))

a.extend(b)
len(a)

5

In [103]:
# 10 articles per page x01 -x99 = 9990
print(len(results_list))

12987


In [104]:
extracted_articles = pd.DataFrame.from_dict(results_list)

In [106]:
extracted_articles.iloc[0]

title          Amid COVID-19 concerns and multiple candidates...
description    LIMA (Reuters) -Peru's presidential candidates...
date                                                 Apr 11 2021
Name: 0, dtype: object

In [107]:
extracted_articles.iloc[-1]

title          Privacy lawyers for Google, Intel to appear at...
description                                       no_description
date                                                     no_date
Name: 12986, dtype: object

In [108]:
extracted_articles.iloc[-5]

title          Ethiopian government says troops take two town...
description    Government forces captured two towns from rebe...
date                                                 Nov 21 2020
Name: 12982, dtype: object

In [112]:
extracted_articles.to_csv('./data/reuters/2021_04_11_2021_11_21.csv')

In [114]:
before_nov_2020 = []

In [122]:
#for page_num in range(1001,4999):
#    if(page_num % 100 == 0):
#        # Pages are numbered 1-99
#        continue
#        
#    if(page_num % 10 == 0):
#        print('extracting data from page: ' + str(page_num))
#          
#    url = get_url(page_num)
#    articles = extract_articles(url)        
#
#    before_nov_2020.extend(articles)

    if '2019' in articles[0]['date']:
        break

extracting data from page: 1010
extracting data from page: 1020
extracting data from page: 1030
extracting data from page: 1040
extracting data from page: 1050
extracting data from page: 1060
extracting data from page: 1070
extracting data from page: 1080
extracting data from page: 1090
extracting data from page: 1110
extracting data from page: 1120
extracting data from page: 1130
extracting data from page: 1140
extracting data from page: 1150
extracting data from page: 1160
extracting data from page: 1170
extracting data from page: 1180
extracting data from page: 1190
extracting data from page: 1210
extracting data from page: 1220
extracting data from page: 1230
extracting data from page: 1240
extracting data from page: 1250
extracting data from page: 1260
extracting data from page: 1270
extracting data from page: 1280
extracting data from page: 1290
extracting data from page: 1310
extracting data from page: 1320
extracting data from page: 1330
extracting data from page: 1340
extracti

In [123]:
extracted_articles_2 = pd.DataFrame.from_dict(before_nov_2020)

In [124]:
extracted_articles_2.iloc[-1]

title                                                   no_title
description    Our apologies, the content you requested canno...
date                                                     no_date
Name: 31018, dtype: object

In [125]:
extracted_articles_2.iloc[-5]

title                                                   no_title
description    Our apologies, the content you requested canno...
date                                                     no_date
Name: 31014, dtype: object

In [126]:
extracted_articles_2.iloc[-50]

title                                                   no_title
description    Our apologies, the content you requested canno...
date                                                     no_date
Name: 30969, dtype: object

In [127]:
extracted_articles_2.to_csv('./data/reuters/prior_to_2021_11_21.csv')

In [130]:
extracted_articles_2.iloc[29314] # First actual article

title          White House to release Obama's 2016 budget on ...
description                                       no_description
date                                                     no_date
Name: 29314, dtype: object

In [132]:
# Count of bad requests
len(extracted_articles_2) - 29314

1705

In [133]:
# try again with "Our apologies, the content..." detected